In [15]:
# Install dependencies
%pip install anthropic python-dotenv

In [16]:
# Load env variables
from anthropic import Anthropic
from google.colab import userdata
from IPython.display import Markdown
from anthropic.types import ToolParam
from datetime import datetime, timedelta
from anthropic.types import Message
import json
import os
import shutil
from typing import Optional, List


In [17]:
# 1. Retrieve the key from Colab Secrets
my_anthropic_key   = userdata.get('ANTHROPIC_API_KEY')
my_anthropic_model = userdata.get('CLAUDE_MODEL')


In [18]:

# Create an API client
client = Anthropic(
    api_key=my_anthropic_key,
)

print("Anthropic client initialized successfully!")


Anthropic client initialized successfully!


In [19]:
# define a system prompt
#system_prompt = """
#You are a creative screenwriter for feature films.
#"""
system_prompt = None

In [20]:
# Utility functions

def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(messages, system=None, stop_sequences=[], tools=None):
    params = {
        "model": my_anthropic_model,
        "max_tokens": 1000,
        "messages": messages,
        "stop_sequences": stop_sequences,
    }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])


In [27]:
# Tools and Schemas
def web_search_schema(model):
    return {
        "type": "web_search_20250305",
        "name": "web_search",
        "max_uses": 5,
        "allowed_domains": ["nih.gov"]
    }

In [22]:
def get_tool_use_block(response):
    """
    Safely searches through Claude's response and returns the first ToolUseBlock.
    Returns None if no tool was called.
    """
    for block in response.content:
        if block.type == 'tool_use':
            return block

    return None

def get_tool_use_blocks(message):
    tool_requests = [
        block for block in message.content if block.type == "tool_use"
    ]

    return tool_requests


In [23]:
def assemble_tool_result(tool_request, tool_output):
  tool_result_block = {
    "type": "tool_result",
    "tool_use_id": tool_request.id,
    "content": json.dumps(tool_output),
    "is_error": False
  }
  return tool_result_block

def assemble_error_tool_result(tool_request, exception):
  tool_result_block = {
    "type": "tool_result",
    "tool_use_id": tool_request.id,
    "content": f"Error: {exception}",
    "is_error": True
  }
  return tool_result_block


In [24]:
def run_tool(tool_name, tool_input):
    if tool_name == "str_replace_based_edit_tool":
        command = tool_input["command"]
        if command == "view":
            return text_editor_tool.view(
                tool_input["path"], tool_input.get("view_range")
            )
        elif command == "str_replace":
            return text_editor_tool.str_replace(
                tool_input["path"], tool_input["old_str"], tool_input["new_str"]
            )
        elif command == "create":
            return text_editor_tool.create(tool_input["path"], tool_input["file_text"])
        elif command == "insert":
            return text_editor_tool.insert(
                tool_input["path"],
                tool_input["insert_line"],
                tool_input["new_str"],
            )
        elif command == "undo_edit":
            return text_editor_tool.undo_edit(tool_input["path"])
        else:
            raise Exception(f"Unknown text editor command: {command}")
    else:
        raise Exception(f"Unknown tool name: {tool_name}")


def run_tools(message):
    tool_requests = [block for block in message.content if block.type == "tool_use"]
    tool_result_blocks = []

    for tool_request in tool_requests:
        try:
            tool_output = run_tool(tool_request.name, tool_request.input)
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": json.dumps(tool_output),
                "is_error": False,
            }
        except Exception as e:
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": f"Error: {e}",
                "is_error": True,
            }

        tool_result_blocks.append(tool_result_block)

    return tool_result_blocks

In [25]:
def run_conversation(messages):
    while True:
        response = chat(
            messages,
            tools=[get_text_edit_schema(my_anthropic_model)],
        )

        add_assistant_message(messages, response)
        print(text_from_message(response))

        if response.stop_reason != "tool_use":
            break

        tool_results = run_tools(response)
        add_user_message(messages, tool_results)

    return messages

In [28]:
messages = []

add_user_message(
    messages,
    """
    What's the best exercise for gaining leg muscle?
    """)
response = chat(messages, tools=[web_search_schema(my_anthropic_model)])
response


Message(id='msg_011CezLV3r1z4JGzCQQMnQmc', container=None, content=[TextBlock(citations=None, text='There\'s no single "best" exercise for leg muscle gain, as the most effective approach typically combines multiple exercises. However, here are some of the most effective options:\n\n**Compound Exercises (Most Effective for Mass):**\n- **Squats** - The most comprehensive leg exercise, working quads, hamstrings, glutes, and core\n- **Deadlifts** - Excellent for posterior chain (glutes, hamstrings, lower back)\n- **Leg Press** - Great for quads and glutes with less spinal stress than squats\n- **Lunges** - Effective for quads, glutes, and hamstrings with good unilateral strength development\n\n**Isolation Exercises (Supplement Compound Moves):**\n- **Leg curls** - Target hamstrings\n- **Leg extensions** - Isolate quadriceps\n- **Calf raises** - Build calf muscles\n\n**Key Principles for Leg Muscle Growth:**\n1. **Progressive overload** - Gradually increase weight or reps over time\n2. **Vo